# Choice logit: model values for the psychometrics

**Question.** Meg's psychometrics use two values: the *true* magnitudes the task offers, and the
*believed* ones (what an arm paid the last time the mouse chose it). Neither is what the mouse
actually uses. This notebook estimates a third, **model value**, from the mouse's own choices, and
redraws the psychometrics and the stay/switch curves with it.

1. **The model.** A logistic regression of choice on port location (a fixed bias per tower),
   reward magnitude history and choice history.
2. **Synthetic data.** Simulated Q-learning mice (same task simulator and value learner as
   `scripts/to_integrate/RL model.ipynb`) whose real values and choice rule we know, and a
   **value-blind control** that repeats its choices and ignores rewards.
3. **Real data.** The same pipeline on the drifting-task cohorts, next to the believed-value
   versions of the figures.

**The trap this notebook is built around.** A model value made of recent rewards is non-zero mostly
for arms the mouse chose recently, and a mouse that repeats its choices picks them again. Ranking
the arms by such a value predicts choice even if the mouse ignores rewards entirely. So **every
curve below comes with a dashed baseline**: what a model with choice history and tower bias but *no
reward term* predicts for the same outcome on the same trials. Only the gap between solid and
dashed is evidence for a value effect. Section 2.5 checks this on an agent that is known to ignore
value.

## 0. Setup and parameters

Everything below is a call into `src/` (`src/pipelines/choice_logit_values.py` and the modules it
uses); the notebook only chooses parameters, calls them, and explains the result.

Real data are loaded **cache-first** from the import cache (`BANDIT_CACHE_DIR`): the newest cached
snapshot of each cohort is used, nothing is written to the data share, and sessions added since
the snapshot are missing (a `StaleCacheWarning` says so). Set `DATA_ROOT` only if the data are not
under `BANDIT_DATA_ROOT`. Figures go to `results/` (git-ignored) as pdf, png and svg with a
`.caption.txt` beside each.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

REPO_ROOT = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src/behavior_import").is_dir()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.behavior_analysis import choice_logit_cv as cv
from src.behavior_analysis import model_value_psychometrics as mvp
from src.behavior_analysis.drifting_trial_table import drifting_trial_table
from src.behavior_visualization import plot_choice_logit as plots
from src.behavior_visualization import plot_null_agents as null_plots
from src.agents import null_agents
from src.pipelines import null_agents as nulls
from src.pipelines import choice_logit_values as pipe
from src.pipelines import regression_extensions as ext
from src.behavior_visualization import plot_regression_extensions as ext_plots
from src.pipelines import switch_regression as switch
from src.behavior_visualization import plot_switch_regression as switch_plots
from src.behavior_analysis import strategy_models
from src.behavior_analysis import stay_by_run_length as stay_rl
from src.agents import kernel_q_agent as kq_agent
from src.pipelines import choice_kernel_long as longk
from src.behavior_visualization import plot_choice_kernel as persist_plots
from src.behavior_visualization import plot_kernel_q as kq_plots
from src.behavior_visualization import plot_second_best as second_plots
from src.behavior_visualization import plot_strategy_models as strategy_plots
from src.pipelines import fitted_q
from src.pipelines import second_best as second
from src.pipelines import strategy_comparison as strategy
from src.settings import get_unchecked_settings


def show(base):
    # figures are saved (pdf, png, svg, caption) by the plot functions; display the png here
    display(Image(filename=f"{base}.png"))

In [ ]:
# ---- parameters (edit here) ------------------------------------------------------------
COHORTS = [1, 3]  # cohorts whose drifting problems are pooled in the real-data section
DATA_ROOT = None  # None: BANDIT_DATA_ROOT. Loading is cache-first and never writes to the share.
OUTPUT_DIR = REPO_ROOT / "results" / "figures" / "choice-logit-values"

N_LAGS = 5  # reward lags and choice lags in the model
TAIL_GRID = pipe.TAIL_GRID  # decays of the reward tail tried (chosen by held-out likelihood)
L2 = 1.0  # penalty on every weight (a Gaussian prior with SD 1 logit)
N_BINS = 5  # per-mouse quantile bins of the continuous value in the psychometrics
MIN_MICE = mvp.MIN_MICE  # a mean from fewer mice is not drawn
POOL_TRIALS = True  # within a mouse, pool the trials of a bin (False: average cells equally)

# Synthetic mice: (learning_rate, beta, kappa = stickiness, plus_bias = bias to plus-shaped ports)
PARAMETER_SETS = [
    (0.30, 1.0, 0.0, 0.0),  # plain Q-learner
    (0.30, 1.0, 1.0, 0.0),  # sticky
    (0.30, 1.0, 0.0, 0.8),  # tower (plus-port) bias
    (0.15, 1.2, 0.5, 0.5),  # slow learner, mild stickiness and bias
    (0.50, 0.8, 1.0, 0.5),  # fast learner, sticky and biased
]
MICE_PER_SET = 2
SYNTHETIC = dict(n_problems=6, sessions_per_problem=4, trials_per_session=200)
SEED = 20261106

# Negative control (section 2.5): ignores reward (beta = 0) but repeats its choices, next to a
# value-driven learner of the same learning rate; three mice of each.
CONTROL_SETS = {"Value-Blind Sticky": (0.30, 0.0, 2.5, 0.0), "Value-Driven": (0.30, 2.0, 1.0, 0.0)}

# Negative controls (sections 2.7 and 3.3): win-stay/lose-shift agents fitted to each mouse.
# The history trace of the agents decays by NULL_DECAY per trial (a half-life of about 7 trials).
NULL_DECAY = null_agents.DEFAULT_DECAY
NULL_SEED = SEED + 2

# Regression extensions (sections 2.8 and 3.4): every model variant is refitted on each of a
# mouse's problems, about 7 s per mouse; mice run in parallel and each mouse's result is cached.
EXT_JOBS = 2  # parallel workers over mice
EXT_BINS = 10  # quantile bins of the calibration figure (deciles)
EXT_CACHE_DIR = get_unchecked_settings().cache_dir / "regression_extensions"  # disposable

# Switch regression (sections 2.9 and 3.5): where a switch goes, beyond choice history and recency.
SWITCH_GRID = switch.sr.TRACE_GRID  # choice-trace decays tried per mouse (held-out likelihood)
SWITCH_CACHE_DIR = get_unchecked_settings().cache_dir / "switch_regression"  # disposable
SWITCH_STRESS = dict(w=1.5, decay=0.9)  # strong, slow recency: the stress case of the controls
SWITCH_SEED = SEED + 3
SWITCH_FIRTH = switch.sr.FIRTH  # Firth's bias reduction in every fit (off: see section 2.9)
# Null calibration: each mouse's own fitted null agents replayed on its task, about 0.6 s per
# replicate and agent (cached). 100 per agent for the mice (about 2 minutes per mouse); 20, and
# WSLS (Magnitude) only, for the synthetic check of the calibration itself.
SWITCH_NULL_REPLICATES = switch.DEFAULT_REPLICATES
SWITCH_NULL_REPLICATES_SYNTHETIC = 20

# Strategy tests (sections 2.10, 3.3 and 3.6). Every fit is cached per mouse (keyed on the code).
# Held-out comparison: three models, each refitted on every leave-one-problem-out fold; the
# Q-learner has STRATEGY_STARTS random starts per fit (about 10 s per fit with 5), the dominant cost.
STRATEGY_JOBS = 2  # parallel workers over mice
STRATEGY_STARTS = strategy_models.Q_STARTS
STRATEGY_STARTS_SYNTHETIC = 2  # two starts (one start of the kernel Q fit can fail to converge): the synthetic learners are well-behaved
STRATEGY_SYNTHETIC_MICE = 3  # mice per synthetic source in the comparison
RECOVERY_SYNTHETIC_MICE = 2  # of those, the mice of the (three times costlier) model recovery
STRATEGY_SEED = SEED + 4
STRATEGY_CACHE_DIR = get_unchecked_settings().cache_dir / "strategy_tests"  # disposable
RECOVERY_REPLICATES = 1  # simulated mice per mouse and model in the model recovery
# Comparison columns of 3.3 to 3.6 (the WSLS agents and the fitted Q-learner) play schedules this many
# times as long as their mouse's (more switches, so no agent drops under the switch regression's
# minimum, and less noise). Calibration nulls and the power check keep the mouse's own length.
CONTROL_LENGTH_FACTOR = 2.0
# Second-best test: null replicates per mouse and agent (as the switch regression's calibration).
SECOND_BEST_REPLICATES = switch.DEFAULT_REPLICATES
SECOND_BEST_REPLICATES_SYNTHETIC = 20
# Power check of the fitted Q-learner (positive control): replicates per mouse, about 1 s each.
POWER_REPLICATES = 50

# Kernel Q-learner and the flat choice kernel (sections 2.11 and 3.7).
KERNEL_LONG_LAGS = longk.LONG_LAGS  # reward and choice lags of the long-lag logit
KERNEL_RECOVERY_RATES = (0.05, 0.1, 0.3)  # generating alpha_k of the known-kernel recovery
# The model recovery refits six models on every simulated mouse (the kernel Q-learner at about 20 s a
# fit, five starts, on every fold): the first RECOVERY_REAL_MICE mice of the real data only (None: all).
RECOVERY_REAL_MICE = None

synthetic_dir, real_dir = OUTPUT_DIR / "synthetic", OUTPUT_DIR / "real"

## 1. The model

On every trial the mouse chooses one of the three offered ports $a$. Each gets a utility

$$
U_a(t) \;=\; \underbrace{b_{\text{tower}(a)}}_{\text{port location}}
\;+\; \underbrace{\sum_{k=1}^{K} w^{r}_k\, r_a(t-k) + w^{\text{tail}}\,\tau_a(t)}_{\text{reward history}}
\;+\; \underbrace{\sum_{k=1}^{K} w^{c}_k\, c_a(t-k)}_{\text{choice history}},
\qquad P(a)=\frac{e^{U_a}}{\sum_{a'} e^{U_{a'}}}
$$

- $r_a(t-k)$ is the reward magnitude (pulses, 0-4) the mouse **received** at port $a$ on trial $t-k$ if
  it chose $a$ then, otherwise 0.
- $\tau_a(t)=\sum_{j>K}\gamma^{\,j-K-1}\,r_a(t-j)$ is the **reward tail**: all rewards older than the
  $K$ lags, decayed by $\gamma$ per trial and summed into one regressor (one extra weight). Without
  it, an arm not chosen for $K$ trials has a model value of exactly 0 although the mouse still has
  a memory of it (a Q-learner keeps an unchosen arm's value indefinitely). $\gamma$ is chosen from
  `TAIL_GRID` by held-out likelihood (section 2.1).
- $c_a(t-k)$ is 1 if $a$ was chosen on trial $t-k$, otherwise 0. A positive $w^c_1$ is a tendency
  to repeat (perseveration).
- $b_{\text{tower}}$ is one fixed bias per tower (the 3x3 field has nine). Adding the same number to
  every tower's bias changes no probability, so the alphabetically first tower is fixed at 0.

It is a *conditional logit*: one set of weights per mouse, a softmax over the three offered ports,
fitted by maximum likelihood with a small L2 penalty (`L2`; a Gaussian prior with SD 1 logit on every
weight, weak next to thousands of trials) and L-BFGS-B with an exact gradient. A fit that does not
converge raises. The tower biases are only comparable if the problems share towers; the design warns
if the problems fall into groups that share none.

**History rules.** Reward history (lags and tail) carries over between the sessions of a problem and
starts empty at a problem's start, like the values of the Q-learner. Choice history is cleared at a
session start and after a dropped trial, as that learner's stickiness is. A lag that reaches back
before the start of the history is zero.

**Model value.** Only the reward part is called value:

$$\hat V_a(t) \;=\; \sum_{k=1}^{K} w^{r}_k\, r_a(t-k) + w^{\text{tail}}\,\tau_a(t).$$

Perseveration (choice weights) and port preference (tower biases) are *nuisance* terms: they are in
$U$, so the fit does not credit the reward kernel for them, but they are left out of $\hat V$.

**Why held-out.** For each problem of a mouse the weights come from a fit to the mouse's **other**
problems (leave-one-problem-out), and $\hat V$ on that problem is computed with them, so no value
is built from weights fitted to the trials it describes.

**The baseline.** Fitted the same held-out way: choice history + tower bias, no reward term. Its
choice probabilities are averaged into the same bins as the observed choices, so a solid curve that
rises more than its dashed baseline is a value effect beyond stickiness and port bias.

**Model comparison** (held out): reward history only; + choice history; + tower bias (the full
model). Scores are negative log likelihood per choice in nats, against $\log 3 \approx 1.0986$
for choosing at random.

**Bins.** The continuous value is cut into `N_BINS` bins of equal numbers of a mouse's trials (ranks,
with ties ordered at random, so a bin is never empty even when many values are exactly equal). Each
bin is placed on the x axis at its mean value (averaged over mice), so the axis is in logit units, and
a mean drawn from fewer than `MIN_MICE` mice is left out. Values that are discrete by nature
(task and believed magnitudes) are plotted at their exact values.

## 2. Synthetic data: Q-learning mice with known values

`simulate_cohort` makes agents of the value learner that the RL notebook and
`src/agents/drifting_simulation.py` use: a softmax over learned values $Q$ with inverse
temperature $\beta$, a stickiness bonus $\kappa$ on the previous choice, and a bias
`plus_bias` for the ports in the same row or column as the initiation port (the known
tower bias), after each choice $Q_a \leftarrow Q_a + \alpha\,(r - Q_a)$. They play the drifting
task (drift attempt rate 0.1, blocks of 30-35 trials, the most chosen arm drops to 0 at a block
change) on six different triples of towers, so every tower is offered in at least two problems and
its bias can be learned from the others. The start magnitudes of each problem are dealt to the ports
at random. The five parameter settings below are each run by two mice.

In [ ]:
synthetic = pipe.simulate_cohort(
    PARAMETER_SETS, mice_per_set=MICE_PER_SET, seed=SEED, **SYNTHETIC
)
print(f"{synthetic.trials['mouse'].nunique()} mice, {len(synthetic.trials):,} trials")
synthetic.parameters

### 2.1 The reward tail, and how many arms have a value

First the decay of the tail, chosen from `TAIL_GRID` by held-out likelihood (lower nats per choice
is better; the first row, with no tail, is for comparison). Then the fraction of (trial, arm) model
values that are exactly 0, without and with the tail.

In [ ]:
selection = pipe.select_tail_decay(synthetic.trials, TAIL_GRID, n_lags=N_LAGS, l2=L2)
analysis = selection.analysis
no_tail = pipe.analyse_trial_table(synthetic.trials, n_lags=N_LAGS, tail_decay=None, l2=L2)
print(f"chosen tail decay: {selection.decay}")
print(
    f"fraction of arm values exactly 0: {mvp.zero_fraction(no_tail.values):.3f} without the tail, "
    f"{mvp.zero_fraction(analysis.values):.3f} with it"
)
selection.table.round(4)

### 2.2 Fit, and kernels against what the generating rule implies

The Q-learner is not exactly a lagged-reward logit, so "recovered" needs a precise target. A reward
that is followed by $m$ further visits to the same arm adds $\beta\alpha(1-\alpha)^m$ to that
arm's utility (an unchosen arm keeps its value, a chosen arm forgets by a factor $1-\alpha$ per
visit). For a reward $k$ trials back, $m$ lies between 0 (never chosen again) and $k-1$ (chosen on
every trial since), so the reward's contribution at lag $k$ lies between

$$\beta\alpha(1-\alpha)^{k-1}\quad\text{and}\quad\beta\alpha .$$

The table gives these bounds as the scale to expect; the fitted weights need not fall inside them,
because the choice-history features and the tail share part of the effect. The **reference** (dashed
in the figure, `reference` in the table) is sharper: the weights of the logit's own features that
come closest, in cross-entropy, to each agent's *generating choice probabilities* (a logit fitted to
those probabilities as soft targets: what a logit would find with unlimited data from the agent,
no sampling noise). The tower biases of the generating rule are $0$ or `plus_bias` for each tower
(relative to the reference tower).

In [ ]:
reference = pipe.reference_weights(analysis, synthetic)
base = synthetic_dir / "Choice Logit Weights"
plots.plot_kernels(
    analysis.weights, base, reference=reference, reference_label="Generating Rule (Soft-Target Fit)"
)
show(base)
pipe.kernel_comparison(analysis, synthetic).round(3).query("lag <= 3")

**Reading the figure.** Thin solid lines are the fitted weights of each mouse, dashed lines the
reference of the same mouse, the thick line their means. Compare the solid and dashed means: the
fitted reward kernel is in the range of the reference, noisier than it, and the tower biases in the
right panel fall near the identity line.

### 2.3 Model comparison

Held-out negative log likelihood per choice: choice history and tower bias should help exactly for
the mice that have stickiness and bias.

In [ ]:
base = synthetic_dir / "Model Comparison"
plots.plot_model_comparison(analysis.comparison, base)
show(base)
gains = analysis.comparison.pivot(index="mouse", columns="model", values="gain_over_chance")[list(cv.MODELS)]
gains.join(synthetic.parameters.set_index("mouse")[["kappa", "plus_bias"]]).round(3)

### 2.4 Psychometrics: model value against the true values, with baselines

For each trial the three arms are ranked by value, and the curves show how often the mouse chose the
best and the second-ranked arm against the best value, the second value, and their difference.
Four rows, all computed on **the same trials of the same mice**:

1. the **model value** $\hat V$ (held out, from the logit),
2. the agents' **own Q-values** (the generating values),
3. the task's **true magnitudes** the ports offer,
4. the **believed values** (the last reward each arm paid; trials where every arm has been chosen
   once in the problem).

Every row has the dashed baseline: the choice-history + tower-bias model's probability of the same
outcome, binned the same way. Rows 3 and 4 are discrete (pulses), so they use exact values; rows 1
and 2 use quantile bins.

In [ ]:
q_values = pipe.generating_values(analysis, synthetic)
bases = dict(
    model=(None, "Model Value", "Logit", N_BINS),
    q=(q_values, "Agent Q-Value", "Pulses", N_BINS),
    task=(pipe.task_values(analysis), "Task Magnitude", "Pulses", None),
    believed=(pipe.believed_values(analysis), "Believed Value", "Pulses", None),
)
rows = []
for values, name, unit, n_bins in bases.values():
    curves, baselines = pipe.value_curves(analysis, n_bins, values, MIN_MICE)
    rows.append(
        plots.PsychometricRow(name, curves, plots.value_labels(name, unit, binned=n_bins is not None), baselines)
    )
base = synthetic_dir / "Psychometric Curves"
plots.plot_psychometric_rows(
    rows,
    "Psychometric Curves: Model, Q, Task and Believed Values",
    (
        "Rows rank the three arms by the model value (held-out logit), by the simulated agents' own "
        "Q-value, by the magnitude each port offers, and by the last reward each arm paid.",
        f"Rows 1 and 2 use per-mouse quantile bins ({N_BINS}); rows 3 and 4 use the exact values.",
    ),
    base,
    MIN_MICE,
)
show(base)

**How close is $\hat V$ to the generating $Q$?** Within a trial only differences between arms matter, so
compare the trial-centred values (each arm minus the trial's mean over the three arms): Spearman
correlation per mouse. The believed value is the baseline to beat: it needs no model at all. Both are
compared on the trials where every arm has a believed value.

In [ ]:
agreement = pipe.value_agreement(analysis, synthetic).set_index("mouse")
display(agreement.round(3).T)
print(
    "mean over mice: model value %.3f, believed value %.3f"
    % (agreement["model_value"].mean(), agreement["believed_value"].mean())
)

### 2.5 Stay/switch curves and switch targets on model values

P(stay) and P(switch) against the value of the arm chosen on the previous trial and of its best
alternative. A continuous value has no "chose the 0-value option" row, so the rows are whether the
previous choice was **rewarded** (a non-zero magnitude) or not. Left to right: the difference
$V(\text{chosen}) - V(\text{alt})$, $V(\text{chosen})$ and $V(\text{alt})$. The same figure for the
model value, the agents' Q-values and the believed values, each with its dashed baseline.

The **switch-target** figure asks where a switch goes. Only the pairs whose next choice is a switch
count; y is the fraction of those switches that go to the *best alternative* (the higher-valued of the
two arms not chosen before), against the gap $V(\text{best alt}) - V(\text{other arm})$ between those
two arms. A switch between two arms of equal value has no best alternative and is left out (counted in
the caption). One panel per reward row, a dotted line at 0.5 (a random pick), and the nuisance
baseline: the choice-history + tower-bias model's own P(best alternative | switch). A value that
guides the switch should rise above 0.5 with the gap, and above its baseline.

In [ ]:
for key, label, unit in (("model", "Model Value", "Logit"), ("q", "Agent Q-Value", "Pulses"), ("believed", "Believed Value", "Pulses")):
    values, _, _, n_bins = bases[key]
    pairs = pipe.value_pairs(analysis, values)
    curves, baselines = mvp.stay_switch_curves_by_reward(
        pairs, pool_trials=POOL_TRIALS, n_bins=n_bins, min_mice=MIN_MICE
    )
    base = synthetic_dir / f"Stay-Switch Curves ({label})"
    plots.plot_stay_switch_by_reward(
        curves, label, POOL_TRIALS, base,
        baselines_by_row=baselines, unit=unit, min_mice=MIN_MICE, binned=n_bins is not None,
    )
    show(base)

    target_curves, target_baselines = mvp.switch_target_curves_by_reward(pairs, n_bins=n_bins, min_mice=MIN_MICE)
    base = synthetic_dir / f"Switch Target Curves ({label})"
    plots.plot_switch_targets_by_reward(
        target_curves, label, mvp.tied_switches_by_reward(pairs), base,
        baselines_by_row=target_baselines, unit=unit, min_mice=MIN_MICE, binned=n_bins is not None,
    )
    show(base)

### 2.6 Negative control: an agent that ignores value

`Value-Blind Sticky` has $\beta=0$: its choices do not depend on $Q$ at all, only on repeating the
previous choice. `Value-Driven` is a Q-learner with the same learning rate. If the raw psychometric
of the model value were evidence of value use, both would show it. The number below is the rise of
P(best) from the lowest to the highest bin of the best value, once as drawn ("raw") and once as the
mean over mice of the observed-minus-baseline rise ("beyond baseline", `value_effect`): the value
effect with stickiness and tower bias subtracted.

In [ ]:
control_rows, control_effects = [], []
for name, parameters in CONTROL_SETS.items():
    cohort = pipe.simulate_cohort([parameters], mice_per_set=3, seed=SEED + 1, **SYNTHETIC)
    result = pipe.analyse_trial_table(cohort.trials, n_lags=N_LAGS, tail_decay=analysis.tail_decay, l2=L2)
    curves, baselines = pipe.value_curves(result, N_BINS, None, MIN_MICE)
    best = curves["best"]["best_val"].mean
    control_effects.append(
        dict(agent=name, raw_rise=best[-1] - best[0], rise_beyond_baseline=mvp.value_effect(curves, baselines))
    )
    control_rows.append(plots.PsychometricRow(name, curves, plots.value_labels("Model Value", "Logit"), baselines))
base = synthetic_dir / "Negative Control"
plots.plot_psychometric_rows(
    control_rows,
    "Negative Control: Value-Blind Sticky Agent and a Value-Driven Agent",
    ("Model-value psychometrics of three mice of each kind; the dashed baseline is the held-out "
     "choice-history + tower-bias model.",),
    base,
    MIN_MICE,
)
show(base)
pd.DataFrame(control_effects).set_index("agent").round(3)

For the value-blind agent the solid curve rises (recently chosen arms are the ones with a model value,
and it repeats them) but the dashed baseline rises with it, so the effect beyond the baseline is about
0. The value-driven agent keeps a positive effect beyond its baseline. This is a conservative number:
the held-out baseline absorbs part of what the value explains, since a value-driven agent also
repeats rewarded choices.

### 2.7 Negative controls: win-stay/lose-shift agents fitted to each mouse

The Q-learner of the sections above learns values, so its reward kernel, its value psychometrics and
its stay/switch curves are what a value learner looks like to the logit. To read the same figures for
the mice we also need agents that have **no value-guided switching at all**, run through exactly the
same pipeline. Four soft win-stay/lose-shift (WSLS) agents (`src/agents/null_agents.py`):

| Agent | P(stay) after the previous choice | Where a switch goes |
|---|---|---|
| **WSLS** | $\sigma(a_\text{win})$ after any reward, $\sigma(a_\text{lose})$ after none; magnitude ignored | uniform over the two other arms |
| **WSLS (Recency)** | the same | first other arm with $\sigma(w\,(h_1-h_2))$, $h$ = decayed count of past choices |
| **WSLS (Magnitude)** | $\sigma(a_\text{win}+b\,(m-1))$ after $m\ge1$ pulses, $\sigma(a_\text{lose})$ after none | as WSLS (Recency) |
| **WSLS (Graded)** | $\sigma(w_0+w_m\,m)$, $m$ the previous magnitude (0 if unrewarded) | the more recently chosen of the two alternatives in this problem with probability $p_\text{recent}$, otherwise the other (50/50 if neither was chosen yet) |

None of them looks at what the *other* arms offer, so none can switch towards the better arm for
value reasons. WSLS (Magnitude) is the realistic one: magnitude grades *staying* (the mice's reward
kernel is graded), but the switch target stays a recency effect. $b=0$ recovers WSLS. WSLS (Graded) replaces the decaying trace by a recency rule that needs no decay constant: $w_0$ and $w_m$ come from a two-parameter logistic regression and $p_\text{recent}$ in closed form from the switches that had a more recent alternative.
The first trial of a session, and the first after a dropped trial, is uniform and clears the trace,
as the logit's choice history is cleared. There is no tower bias: the agents have no port
preference (so the logit's baseline has less to absorb than for a mouse).

**A limit of the switch-target comparison.** The logit's choice-history baseline sees five choice
lags and no choice tail. The agents' trace decays by 0.9 per trial, so most of it lies beyond those
five lags and the baseline cannot absorb where a recency-guided switch goes: recency alone gives
above-baseline switch targets (section 2.7 shows this, with a fast-decaying trace that fits in five
lags as the control). The mice's perseveration is also long, so their own excess may be inflated by the
same blind spot. The switch regression of the next analysis controls for it with a long choice trace
and trials-since-chosen.

Each agent is **fitted to every mouse** (the stay logits in closed form or by a two-parameter
logistic fit, $w$ by a one-dimensional maximum likelihood), then **plays that mouse's own problems**:
the same towers, initiation port, starting magnitudes and number of trials per session, in a task
whose drift and block lengths are measured from the data. The magnitudes are generated by the task
rules, not replayed: a block change zeroes the arm the *agent* chose most, so the mouse's recorded
schedule belongs to the mouse's own choices. The simulated agents then go through the same logit
analysis as the mice. Every agent's likelihood is scored **leave-one-problem-out**, on the same folds as the
choice logit (fitted on all but one problem of the mouse, scored on the held-out one, segment
starts credited log(1/3) so that nats per trial are comparable); the in-sample score is
kept in `comparison.in_sample`, labeled, and is not compared with the logit.

The helper below does all of that for one trial table and one reference analysis (the Q-learners
here, the mice in section 3.3), writing figures to a `negative-controls/` subfolder.

In [ ]:
def run_null_controls(
    trial_table, reference, reference_label, out_dir, extra_cohorts=None, rules=None, length_factor=1.0
):
    """Fit the null agents to the mice of *reference*, play them, tabulate, draw and show.

    *extra_cohorts* (``{label: SyntheticCohort}``, e.g. the fitted Q-learner) are analyzed and
    drawn like the agents; *rules* are the task rules of the simulations (default: measured);
    *length_factor* is the schedule length of the simulated agents relative to the mouse's.
    """
    comparison = nulls.compare_with_controls(
        trial_table, reference, reference_label, NULL_SEED,
        n_bins=N_BINS, min_mice=MIN_MICE, pool_trials=POOL_TRIALS, decay=NULL_DECAY,
        rules=rules, extra_cohorts=extra_cohorts, length_factor=length_factor,
    )
    null_plots.plot_comparison(comparison, out_dir)
    for kind, label in nulls.AGENT_LABELS.items():
        print(f"fitted parameters, {label}")
        display(nulls.parameters_table(comparison.controls.fits[kind]).round(3))
    print("negative log likelihood per choice (leave-one-problem-out, agents and logit)")
    display(comparison.scores.pivot(index="mouse", columns="model", values="nats_per_choice").round(4))
    print("rise of P(best) from the lowest to the highest bin of the best value")
    display(comparison.effects.set_index("source").round(3))
    print("switch targets: observed minus choice baseline, by reward row")
    display(comparison.switch_excess.round(3))
    for name in ("Null Agent Fits", "Kernels by Source", "Psychometric Curves"):
        show(out_dir / name)
    for label in [*nulls.AGENT_LABELS.values(), *(extra_cohorts or {})]:
        show(out_dir / f"{label} Stay-Switch Curves")
        show(out_dir / f"{label} Switch Target Curves")
    return comparison

**Run on the Q-learners.** Reference: the five parameter settings of section 2 (the analysis of
section 2.1, with its tail decay). Each null agent is fitted to a simulated Q-learner and plays
generated schedules matched to that mouse (its towers, initiation port, starting magnitudes and
trials per session, with the drift and block lengths measured from the data); read the null agents'
rows against the Q-learner's. Every likelihood is held out on the same folds.

In [ ]:
null_synthetic = run_null_controls(synthetic.trials, analysis, "Q-Learner", synthetic_dir / "negative-controls")

**Reading the figures** (numbers from the run above, seed `SEED`).

- *Kernels by source.* The Q-learner's reward kernel is positive over all five lags (about 0.26,
  0.22, 0.13, 0.12, 0.09). The win-stay/lose-shift agents have a lag-1 weight (WSLS and
  WSLS (Recency) about 0.33, WSLS (Magnitude) about 0.55) and about zero from lag 2 on.
- *Likelihood.* Every score is leave-one-problem-out, on the same folds. The logit has the lowest
  negative log likelihood per trial for all ten mice (for example 0.731 against 0.832 for WSLS
  (Magnitude) and 0.836 for WSLS (Graded) in sim01). WSLS (Magnitude) is the best agent for nine of
  ten mice and WSLS (Graded) for one (sim09); WSLS (Graded) is within 0.01 of WSLS (Magnitude) for
  all ten mice. The two do not nest, so there is no guaranteed order any more; WSLS and WSLS (Recency)
  are always the two worst (0.906 and 0.888 in sim01).
- *Psychometrics.* The rise of P(best) from the lowest to the highest bin beyond the choice baseline
  is 0.078 for the Q-learner and 0.056, 0.051, 0.137 and 0.175 for WSLS, WSLS (Recency), WSLS
  (Magnitude) and WSLS (Graded): it is not zero for the null agents. The baseline has no reward term, so staying after a
  reward (and, more, after a large reward) looks like a value effect. Read the Q-learner's and the
  mice's rise against these numbers, not against zero.
- *Switch targets.* The table gives observed minus baseline, averaged over bins (rewarded row,
  unrewarded row): Q-learner 0.096, 0.089; WSLS 0.006, 0.012; WSLS (Recency) 0.032, -0.004; WSLS
  (Magnitude) 0.041, 0.039; WSLS (Graded) 0.001, -0.012. At the fitted switch weights (w about 0.2) the null agents sit close to
  the baseline. That is not a property of recency in general: with a strong recency weight (w = 1.5)
  and the default decay 0.9, the rewarded-row excess is about 0.30 in a test run
  (`tests/test_null_agents_pipeline.py`), and it is about 0 when the trace decays by 0.5 (it then
  fits in the baseline's five lags). The cause is the baseline's five choice lags without a choice tail, not
  value guidance. The same test shows the same pattern for WSLS (Magnitude): its excess is about
  0.30 at w = 1.5 and decay 0.9 and about 0 at w = 0 (graded staying alone) or at decay 0.5, so the
  small excess of the fitted agent is the recency effect and not graded staying. The switch regression
  of the next analysis controls for it with a long choice trace and trials-since-chosen.

### 2.8 Regression Extensions

Four questions about the choice logit, each asked of the same five sources so that the mice can be
read against a positive and three negative controls: the **Q-learners** above (they use reward
magnitudes and keep a value for every arm) and the three **win-stay/lose-shift agents** of 2.7
(no value for unchosen arms, magnitude only in WSLS (Magnitude)). Every number is held out:
a model is fitted on all but one problem of a mouse and scored on the problem it did not see, in
nats per choice (lower is better; chance is log 3 = 1.0986).

1. **Believed value.** The magnitude an arm last paid when the mouse chose it in the current
   problem, plus a **never-visited indicator** (1 before the arm's first visit, where the value is
   set to 0), so an unvisited arm is neither worth 0 nor dropped. Variants: (a) the full model,
   (b) full + believed value, (c) choice history + tower bias + believed value, without reward
   lags. The weights come from (b) fitted to all of a mouse's trials. If (b) is no better than (a),
   the believed value adds nothing to the reward lags.
2. **Binary against pulse.** The reward lags code pulses (the standard model). *Binary* codes 1 if
   rewarded and 0 otherwise, with the same lags and tail; *binary + pulse* has both. WSLS and WSLS
   (Recency) cannot see magnitude and should prefer binary; WSLS (Magnitude) and the Q-learner
   should prefer pulse. The binary model's tail decay is chosen by its own held-out likelihood, as the pulse model's is.
   With fewer than 6 mice per source the figure shows mice individually, with no test.
3. **Unique and shared variance.** For tower bias, reward history and choice history: the unique
   variance of a group is the held-out cost of dropping it from the full model. The shared term is
   `[gain(reward + tower) + gain(choice + tower)] - gain(full)` with gains over the tower-only model,
   so unique reward + unique choice + shared = the full model's gain over the tower-only model
   (single-group models are in the tables too).
4. **Calibration.** Every (trial, arm) pair is a prediction; predictions are cut into quantile bins
   and the y axis is the **observed fraction of times that arm was chosen**, against the mean
   predicted probability. One panel for the full model, one for the model with choice history only.

Figures go to `regression-extensions/` next to the other figures of each section.

In [ ]:
def report_regression_extensions(sources, out_dir):
    """Tabulate the four analyses of each source, draw the figures, show them and the summaries."""
    tables = {label: ext.extension_tables(source, EXT_BINS) for label, source in sources.items()}
    ext_plots.plot_extensions(tables, out_dir)
    summaries = ext.source_summaries(tables)
    print("held-out gains in nats per choice (positive: better than the full pulse model, or, for the")
    print("believed-value rows, than the full model); a source with fewer than 6 mice has no mean or p-value")
    display(summaries.gains.round(4))
    print("believed-value and never-visited weights (logit per pulse, logit)")
    display(summaries.weights.round(3))
    print("unique and shared variance, held-out nats per choice")
    display(summaries.partition.round(4))
    print("calibration: mean over bins of |observed - predicted|, weighted by bin size, the noise floor")
    print("of each (a perfectly calibrated model's error), and held-out nats per choice")
    display(summaries.calibration.round(4))
    print("binary tail decay chosen per source:", {k: t.binary_tail_decay for k, t in tables.items()})
    print("every mouse (read the sources with fewer than 6 mice here)")
    display(ext.per_mouse_summary(tables).round(4))
    for name in ("Believed Value", "Binary vs Pulse", "Unique and Shared Variance"):
        show(out_dir / name)
    for label in sources:
        show(out_dir / f"Calibration - {label}")
    return tables, summaries

In [ ]:
ext_synthetic = ext.analyse_sources(
    ext.comparison_inputs("Q-Learner", synthetic.trials, null_synthetic), EXT_JOBS, EXT_CACHE_DIR
)
tables_synthetic, summaries_synthetic = report_regression_extensions(
    ext_synthetic, synthetic_dir / "regression-extensions"
)

**Reading the figures** (numbers from the run above, seed `SEED`; 10 mice per source; gains in held-out nats per choice, mean ± se, positive is better).

- *Believed value.* Adding it to the full model helps the Q-learner (+0.0125 ± 0.0023, all 10 mice, Wilcoxon p = 0.002) and nothing else (WSLS +0.0001, Recency +0.0004, Magnitude -0.0001, none significant). The Q-learner's believed-value weight is 0.30 ± 0.04 logit per pulse, the agents' 0.02 to 0.07. Believed value *instead of* the reward lags is no worse than the lags for the Q-learner (-0.0073 ± 0.0050, p = 0.23) but costs the null agents 0.006 to 0.025 (p = 0.002). The never-visited weight is noisy (Q-learner -0.26 ± 0.12, agents about 0 to +0.1).
- *Binary against pulse.* The binary model's tail decay is now chosen by its own held-out likelihood (0.7 for the Q-learner, 0.5, 0.85 and 0.85 for WSLS, Recency and Magnitude); the gains hardly moved. WSLS and WSLS (Recency) prefer binary in all 10 mice (+0.0168 and +0.0152); WSLS (Magnitude) and the Q-learner prefer pulse in all 10 (-0.0285 and -0.0270). Adding binary lags to pulse lags gains +0.0166 and +0.0144 for the two magnitude-blind agents and +0.0010 and +0.0013 for the other two. The binary columns are penalized more than the pulse columns for the same effect (see the figure caption), which slightly favors pulse; the split between the sources is far larger than that.
- *Unique and shared variance.* Tower bias has no unique variance anywhere (-0.004 to +0.003). Unique reward is 0.077 for the Q-learner and 0.017, 0.017 and 0.057 for WSLS, Recency and Magnitude; unique choice is 0.005, 0.034, 0.039 and 0.011. The shared term dominates (0.36, 0.24, 0.24, 0.27): a rewarded choice is also a repeated choice, so reward and choice lags explain the same stays.
- *Calibration.* Calibration is not predictive quality. Mean |observed - predicted| across deciles of the full model is 0.014 (Q-learner), 0.027, 0.026 and 0.009 (WSLS, Recency, Magnitude), against noise floors of 0.007 to 0.008: the full model is miscalibrated beyond sampling noise for the Q-learner and the two magnitude-blind agents. The choice-history-only model is at its floor (0.007, 0.008, 0.012 and 0.008 against 0.007 to 0.008; only Recency is above), yet it predicts worse (held-out nats per choice 0.739, 0.827, 0.824, 0.829 against 0.663, 0.812, 0.808, 0.774 for the full model): it has few distinct predictions, so its bins pool similar situations and match their mean by construction.

### 2.9 Switch Regression

**Question** (meeting to-do). On trials where the mouse leaves its previous arm, does the value of
the two alternatives predict which one it goes to, beyond choice history, recency and tower bias?
Separate models after an unrewarded (*after zero*) and a rewarded (*after non-zero*) previous choice.

**Why new controls.** Section 2.7 showed that the choice baseline of the logit (five choice lags, no
tail) cannot absorb a slow recency trace: WSLS (Recency) with $w=1.5$ and decay 0.9 lands about 0.3
above that baseline on switch targets with no value guidance at all. The switch regression therefore
adds the null agents' own kind of trace and a trials-since-chosen term:

$$
U(a) = \beta_V V(a) + \sum_{k=2}^{K}\beta_k c_a(t-k) + \beta_{tr} h_a(t)
+ \beta_{rec}\log\bigl(1+\min(\text{trials since } a \text{ chosen}, 100)\bigr)
+ \beta_{nev}\,[a \text{ never chosen in the problem}] + b_{\text{tower}(a)},
$$

$P(\text{first}) = \sigma(U(\text{first}) - U(\text{second}))$ over the two arms other than the
previous choice. Lag 1 is the previous choice, never in the choice set, so lags start at 2. $h_a$ is
the decayed count of past choices within the segment; its decay is chosen per mouse from
`SWITCH_GRID` (it includes 0.9 and 0.95) by the held-out likelihood of the history-only model and
used for every value variant (chosen without the value, so for a mouse whose switches follow
value the trace can take up part of it: a real weight is biased towards 0, and the held-out gain is
likely biased down). No intercept: the order of the two alternatives cannot matter; port
preferences are the tower biases.

Three value variants, fitted separately: **true** (the magnitude the arm offers on the switch
trial), **believed** (what it paid when last chosen in the problem) and **model** (the logit's
held-out reward part; a *consistency check*, not an independent test, because it is built from the
same reward history). For each, the value weight $\beta_V$ (logit per pulse; per logit for the
model value) and the held-out gain over the history-only model (leave-one-problem-out, nats per
switch).

**Readouts.** The believed value and the true value *after non-zero*. The true value after zero
is weak even for the Q-learner (it follows what it learned, not the drifting magnitude) and cannot
separate a value learner from the null agents at these switch counts.

**Validation, and why the mice are read against their own null.** The win-stay/lose-shift agents
never look at the alternatives' values, so their value weight should be about 0 and adding the
value should not help held out. On average it is, at realistic switch counts (300-700 per
condition). But a single agent's raw weight is not reliably 0: with a strong, nearly deterministic
recency rule, and block changes that zero the arm chosen most, the true value is correlated with the
trace, and the null weights of a strong recency agent spread with an SD of 0.1-0.3 logit per pulse
(more than the Q-learner's true-value weight after zero), with a bias of about -0.1 near the
100-switch minimum. Firth's bias reduction (`SWITCH_FIRTH`) narrows that spread by 9-40% for the
stress agents and by 4-11% for agents with realistic fitted weights, and leaves the bias, at 3.3
times the cost, so it is off by default. Instead every mouse is **calibrated against its own null**:
its fitted WSLS (Magnitude) and WSLS (Recency) agents are replayed on its own task many times and
go through the identical analysis; the mouse's weight is reported as a calibrated z and an
empirical p, and drawn with a gray band, the middle 95% of its WSLS (Magnitude) null. Below, the
calibration itself is checked on "mice" whose rule is known: Q-learners (should land beyond their
null for the believed value) and two sets of null agents (calibrated p should spread over 0-1). Sources: the
Q-learners of section 2, the three agents fitted to them (2.7, reused) and two stress agents with
$w = 1.5$, decay 0.9 (WSLS (Recency), and WSLS (Magnitude) with graded staying), played on the same
layouts. The stress agents have no logit analysis here, so their model-value panels say "not
fitted". The model value is not calibrated either (each replicate would need its own choice logit).

In [ ]:
def report_switch_regression(sources, out_dir, calibrations=None):
    """Draw the switch-regression figures and show the summaries across mice and every mouse."""
    switch_plots.plot_switch_regression(sources, out_dir, calibrations)
    summary = switch.source_summaries(sources)
    print("raw value weight (beta) and held-out gain of the value over the history-only model;")
    print("a source with fewer than 6 mice has no mean, se or test")
    print("(re_*: random-effects summary of beta)")
    table = switch.per_mouse_table(sources)
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        display(summary.round(4))
        print("every mouse (switch counts and the chosen trace decay included)")
        display(table.round(4))
        for label, calibration in (calibrations or {}).items():
            print(f"null calibration of {label} ({calibration.replicates} replicates per agent)")
            display(calibration.calibration.round(4))
            display(calibration.summary.round(4))
    for name in switch_plots.FIGURE_NAMES:
        show(out_dir / name)
    return summary, table


def stress_cohorts(trial_table, seed):
    """WSLS (Recency) and WSLS (Magnitude) with a strong, slow recency trace on these layouts."""
    mice = sorted(trial_table["mouse"].unique())
    w, decay = SWITCH_STRESS["w"], SWITCH_STRESS["decay"]
    agents = {
        "WSLS (Recency) w=1.5": (
            "wsls_recency", null_agents.WslsRecencyParams(1.0, -0.5, w, decay)
        ),
        "WSLS (Magnitude) w=1.5": (
            "wsls_magnitude", null_agents.WslsMagnitudeParams(0.2, -0.5, 0.5, w, decay)
        ),
    }
    cohorts = {
        label: nulls.simulate_null_cohort(trial_table, dict.fromkeys(mice, p), kind, seed)
        for label, (kind, p) in agents.items()
    }
    return {label: (cohort.trials, None) for label, cohort in cohorts.items()}


def calibration_check(calibrations):
    """Per source, value and condition: how the calibrated p of the value weight is spread."""
    rows = []
    for label, calibration in calibrations.items():
        betas = calibration.calibration.query("term == 'beta'")
        summary = betas.groupby(["value", "condition"], sort=False).agg(
            n_mice=("p_empirical", "size"),
            mean_p=("p_empirical", "mean"),
            share_beyond_all_nulls=("p_empirical", lambda p: (p <= p.min() + 1e-12).mean()),
            share_p_below_0_1=("p_empirical", lambda p: (p < 0.1).mean()),
            mean_z=("z", "mean"),
        )
        rows.append(summary.reset_index().assign(source=label))
    return pd.concat(rows, ignore_index=True).set_index(["source", "value", "condition"])


switch_inputs = {
    **ext.comparison_inputs("Q-Learner", synthetic.trials, null_synthetic),
    **stress_cohorts(synthetic.trials, SWITCH_SEED),
}
switch_synthetic = switch.analyze_switch_sources(
    switch_inputs, trace_grid=SWITCH_GRID, firth=SWITCH_FIRTH, n_jobs=EXT_JOBS,
    cache_dir=SWITCH_CACHE_DIR,
)

# Calibration of the calibration: "mice" whose rule is known, each against its own fitted
# WSLS (Magnitude) agent. Value learners should land beyond their null; null agents should not.
switch_rules = nulls.task_rules_by_cohort(synthetic.trials)
calibration_kwargs = dict(
    agents=("wsls_magnitude",), replicates=SWITCH_NULL_REPLICATES_SYNTHETIC, rules=switch_rules,
    seed=SWITCH_SEED, n_jobs=EXT_JOBS, cache_dir=SWITCH_CACHE_DIR,
)
null_mice = {
    "WSLS (Magnitude)": null_synthetic.controls.cohorts["wsls_magnitude"].trials,
    "WSLS (Magnitude) w=1.5": switch_inputs["WSLS (Magnitude) w=1.5"][0],
}
switch_calibrations_synthetic = {
    "Q-Learner": switch.calibrate_source(
        synthetic.trials, switch_synthetic["Q-Learner"], null_synthetic.controls.fits,
        **calibration_kwargs,
    ),
    **{
        label: switch.calibrate_source(
            table, switch_synthetic[label],
            {"wsls_magnitude": null_agents.fit_null_agents(table, "wsls_magnitude")},
            **calibration_kwargs,
        )
        for label, table in null_mice.items()
    },
}
print("calibrated p of the value weight: spread for null mice, beyond all nulls for value learners")
display(calibration_check(switch_calibrations_synthetic).round(3))
switch_summary_synthetic, switch_mice_synthetic = report_switch_regression(
    switch_synthetic, synthetic_dir / "switch-regression", switch_calibrations_synthetic
)

**Reading the figures** (numbers from the run above, seed `SEED`; 10 mice per source; mean ± se across mice).

- *Raw weights of the controls.* For all five win-stay/lose-shift sources the mean true- and
  believed-value weights are close to 0 in both conditions (-0.042 to +0.020 logit per pulse, no
  Wilcoxon p below 0.13), and adding the value never improves the held-out fit (gains -0.0025 to
  +0.0007 nats per switch). The stress agents choose decay 0.9 in all 20 mice and their trace weight
  comes out at about 1.5, the generating w. But look at the spread, not only the means: the stress
  agents' single-mouse weights reach ±0.3 (gray bands in the true-value panels), more than the
  Q-learner's true-value weight after zero. A raw weight near 0.1 says nothing on its own.
- *The Q-learner switches towards value.* Believed value 0.45 ± 0.05 logit per pulse after zero and
  0.45 ± 0.04 after non-zero, positive in all 10 mice, held-out gains 0.037 and 0.060; true value
  0.22 ± 0.02 after non-zero. True value after zero is only 0.07 ± 0.03 (gain 0.0006), inside the null
  agents' spread: it is not a usable readout at these switch counts.
- *The calibration works.* Each "mouse" against its own fitted WSLS (Magnitude), 20 replicates
  (table above). Q-learners: the believed value (both conditions) and the true value after non-zero
  lie beyond all 20 nulls in 10 of 10 mice (mean calibrated z 8 to 15); the true value after zero in only
  4 of 10. Null mice, fitted WSLS (Magnitude) and the strong-recency WSLS (Magnitude) w=1.5: mean
  calibrated p 0.41 to 0.66 and 0 to 20% of mice below p = 0.1 per row (10% expected for a calibrated
  test), so the calibration does not turn recency into value even where the raw weights are widest.
- *Model value (consistency check, not calibrated).* The Q-learner's model-value weight is 1.11 ± 0.07
  and 1.47 ± 0.09 per logit (gains 0.020 and 0.057). For the null agents it is noisy (WSLS (Magnitude)
  after non-zero +0.27 ± 0.09, WSLS (Recency) after zero -0.32 ± 0.15) with held-out gains at or below
  0; read it only next to its gain.
- Firth's bias reduction was tested on the null agents at realistic and near-minimum switch counts
  (see `docs/issues/feat-switch-regression.md`): it narrows the stress agents' null spread but does not
  remove their near-minimum bias, so it is off (`SWITCH_FIRTH`) and the calibration does the work.

### 2.10 Strategy Tests

**Questions** (meeting to-do). (1) *The second-best test.* When the mouse leaves the best arm, does it
go to the second best rather than the third? (2) *Heuristic or value learner?* Does a heuristic
that switches towards what the alternatives last paid and towards never-visited arms predict held-out
choices as well as a Q-learner?

**1. Second best** (`src/behavior_analysis/second_best.py`). Same switch trials as 2.9 (the choice
set is the two arms other than the previous choice). *Leave-best*: the previous choice had the strict
maximum magnitude; the outcome is whether the better of the two alternatives was chosen. *Believed
better* and *truly better*: the alternative that last paid more, or that currently pays more. Graded
curves split these by the magnitude difference. Ties are excluded and counted; a mouse with fewer than 30
used switches in a cell is skipped there and named. **Leave-best never follows a zero payout** (the payout
is the chosen arm's magnitude and exactly one arm pays 0), so the after-zero panel is empty by
construction. The 0.5 line is for orientation only: recency alone puts a value-blind mouse above it, so
every mouse is calibrated against its own fitted WSLS (Magnitude), WSLS (Recency) and WSLS (Graded) agents, as in 2.9.

**2. Model comparison** (`src/behavior_analysis/strategy_models.py`). Per mouse, each model is fitted on
all but one problem and scored on the held-out one (nats per choice, segment starts excluded for every
model). *WSLS (Magnitude)*; *Heuristic+*: the same stay rule, with switch logit
$w\,\Delta h + v\,\Delta\text{believed} + u\,\Delta\text{never visited}$ (the trace decay is **fixed
at 0.9**, not fitted, so Heuristic+ nests WSLS (Magnitude) and a difference is due to $v$ and $u$; $u$
has a weak Gaussian prior because mice almost always take the unvisited arm); the *Q-learner*
(`value_agent`, several starts); *WSLS (Graded)* (stay logit linear in the previous magnitude, switches to the more recently chosen alternative with probability $p_\text{recent}$; it does not nest WSLS (Magnitude)). The choice logit's held-out nats are listed as a descriptive reference
only. **Model recovery** plays each model, fitted to each mouse, on that mouse's own problems and asks
whether the comparison picks the model that generated the choices; parameter recovery checks $v$, $u$,
the Q-learner's learning rate and $p_\text{recent}$. If recovery is poor the figure caption says so.

Here: the synthetic Q-learners of section 2 and the three WSLS agents fitted to them (2.7). Second best
on all ten mice per source; the comparison and recovery on the first `STRATEGY_SYNTHETIC_MICE` mice of
the Q-learner and WSLS (Magnitude) sources with `STRATEGY_STARTS_SYNTHETIC` starts, to keep the run to a
few minutes. Runtime of this section: about 150 s on 4 cores with cold caches (the whole synthetic notebook about 11 minutes). Figures go to `strategy-tests/`.

In [ ]:
def report_strategy_tests(second_sources, second_calibrations, comparison_sources, recovery, out_dir):
    """Draw the four strategy-test figures, show them and the tables behind them."""
    second_plots.plot_second_best(second_sources, out_dir, second_calibrations)
    strategy_plots.plot_strategy_models(comparison_sources, out_dir, recovery)
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        print("second best: mean ± se across mice per source (a source with fewer than 6 mice has none)")
        display(second.source_summaries(second_sources).round(3))
        for label, calibration in second_calibrations.items():
            print(f"second best: calibrated z of {label} against its mice's own null agents")
            display(calibration.summary.round(3))
        print("second best: every mouse")
        display(second.per_mouse_table(second_sources).round(3))
        print("model comparison: held-out differences (first minus second, negative: first fits better)")
        display(strategy.source_summaries(comparison_sources).round(4))
        print("model comparison: mice won by each model")
        display(strategy.win_table(comparison_sources).pivot(index="source", columns="model", values="n_mice"))
        print("model comparison: every mouse (held-out nats per choice, winner, fitted parameters)")
        display(strategy.per_mouse_table(comparison_sources).round(4))
        print("model recovery: simulated mice won by each model (rows: generating model)")
        display(recovery.counts)
        display(recovery.fractions.round(2))
        print("parameter recovery of the generating model")
        display(recovery.summary.round(3))
    for name in [*second_plots.FIGURE_NAMES, *strategy_plots.FIGURE_NAMES]:
        show(out_dir / name)

In [ ]:
def first_mice(table, n):
    keep = sorted(table["mouse"].unique())[:n]
    return table[table["mouse"].isin(keep)].reset_index(drop=True)


strategy_dir_synthetic = synthetic_dir / "strategy-tests"
cohorts_synthetic = null_synthetic.controls.cohorts
second_inputs_synthetic = {
    "Q-Learner": synthetic.trials,
    **{nulls.AGENT_LABELS[k]: cohorts_synthetic[k].trials for k in null_agents.KINDS},
}
second_synthetic = second.analyze_second_best_sources(second_inputs_synthetic)
second_calibrations_synthetic = {
    "Q-Learner": second.calibrate_second_best(
        synthetic.trials, second_synthetic["Q-Learner"], null_synthetic.controls.fits,
        agents=("wsls_magnitude",), replicates=SECOND_BEST_REPLICATES_SYNTHETIC,
        rules=switch_rules, seed=STRATEGY_SEED, n_jobs=STRATEGY_JOBS, cache_dir=STRATEGY_CACHE_DIR,
    )
}
compare_kwargs = dict(
    n_starts=STRATEGY_STARTS_SYNTHETIC, n_jobs=STRATEGY_JOBS, cache_dir=STRATEGY_CACHE_DIR
)
compare_inputs_synthetic = {
    "Q-Learner": first_mice(synthetic.trials, STRATEGY_SYNTHETIC_MICE),
    "WSLS (Magnitude)": first_mice(cohorts_synthetic["wsls_magnitude"].trials, STRATEGY_SYNTHETIC_MICE),
}
comparison_synthetic = {
    label: strategy.analyze_comparison_source(table, **compare_kwargs)
    for label, table in compare_inputs_synthetic.items()
}
recovery_source = comparison_synthetic["Q-Learner"]
recovery_mice = sorted(recovery_source.fits)[:RECOVERY_SYNTHETIC_MICE]
recovery_synthetic = strategy.recover_models(
    first_mice(compare_inputs_synthetic["Q-Learner"], RECOVERY_SYNTHETIC_MICE),
    recovery_source._replace(fits={m: recovery_source.fits[m] for m in recovery_mice}),
    rules=switch_rules,
    replicates=RECOVERY_REPLICATES, seed=STRATEGY_SEED, n_jobs=STRATEGY_JOBS,
    cache_dir=STRATEGY_CACHE_DIR,
)
report_strategy_tests(
    second_synthetic, second_calibrations_synthetic, comparison_synthetic, recovery_synthetic,
    strategy_dir_synthetic,
)

**Reading the figures** (synthetic, seed `STRATEGY_SEED`; second best on 10 mice per source, the
comparison on 3 mice per source and the recovery on 2, so those two carry no mean, se or test).

- *Second best.* After a non-zero payout the Q-learners choose the second best in 0.582 ± 0.015 of the
  leave-best switches, the value-blind agents in 0.508 (WSLS), 0.498 (Recency), 0.469 (Magnitude) and
  0.486 (Graded): the agents sit at or below 0.5, so 0.5 is a fair reference here. Believed-better is
  0.71 for the Q-learners and 0.50 to 0.56 for the agents; truly better after non-zero is 0.606
  against 0.47 to 0.51. The after-zero leave-best panel is empty for every source (structural). The
  Q-learners' calibrated z against their own fitted WSLS (Magnitude) is 4.4 ± 0.7 (second best),
  10.3 ± 0.9 (believed-better, after non-zero) and 6.9 ± 0.7 (truly better, after non-zero), all ten
  mice positive (Wilcoxon p 0.002): the test separates a value learner from a recency learner.
- *Model comparison.* On the Q-learner source the Q-learner wins all three mice; Heuristic+ is 0.06 to
  0.09 nats worse, and WSLS (Graded) 0.08 to 0.13 worse than the Q-learner and 0.004 to 0.007 worse
  than WSLS (Magnitude). On the WSLS (Magnitude) source WSLS (Magnitude) wins two mice and Heuristic+
  one by under 0.0002 nats (it nests WSLS (Magnitude), so this is chance); the Q-learner is 0.04 to
  0.07 worse and WSLS (Graded) 0.004 to 0.013 worse.
- *Recovery.* Each of the four models is recovered in every simulated mouse (4 of 4 on the diagonal,
  two mice each, one replicate). The Heuristic+ weight v is recovered to within 0.02 (bias 0.002), the
  never-visited weight u is biased by +0.44 (not explained here), and the correlations are not
  computed from two mice. With two mice this checks that the
  machinery works; the mice's own recovery in 3.6 is the real check.
- *New models.* On the Q-learner source the Q-learner wins all three mice (the kernel Q-learner contains it, so it cannot
  lose by more than its extra parameters); on the WSLS (Magnitude) source WSLS (Magnitude) wins two mice and WSLS (Habit) one,
  by under 0.001 nats. See 2.11 for the kernel fits.


### 2.11 Kernel Q-Learner and the Flat Choice Kernel (synthetic)

The new models of section 3.7 on the synthetic mice: the Q-Learner (Kernel + Forgetting) and WSLS (Habit) are in
the comparison of 2.10 already; here are the ablations, the trace-decay profile, the long-lag kernel and the
run-length curves for the same sources. The synthetic Q-learners have a one-trial stickiness (sharp kernel), so a
slow kernel is not expected from them; the kernel-only agents of the known-kernel recovery are the check that a
slow kernel *would* be seen. Figures go to `kernel-q/`.

In [ ]:
kq_dir_synthetic = synthetic_dir / "kernel-q"
synthetic_mice = sorted(comparison_synthetic["Q-Learner"].fits)
kq_profile_synthetic = strategy.decay_profile_table(
    compare_inputs_synthetic["Q-Learner"], synthetic_mice, n_jobs=STRATEGY_JOBS, cache_dir=STRATEGY_CACHE_DIR
)
long_inputs_synthetic = {"Q-Learner": synthetic.trials, **{nulls.AGENT_LABELS[k]: cohorts_synthetic[k].trials for k in null_agents.KINDS}}
long_synthetic = {
    label: longk.analyze_long_kernel(table, n_lags=KERNEL_LONG_LAGS, n_jobs=STRATEGY_JOBS, cache_dir=STRATEGY_CACHE_DIR)
    for label, table in long_inputs_synthetic.items()
}
kq_timescales_synthetic = strategy.timescale_table(comparison_synthetic["Q-Learner"], long_synthetic["Q-Learner"].weights)
kq_plots.plot_kernel_q_figures(comparison_synthetic, kq_profile_synthetic, kq_timescales_synthetic, kq_dir_synthetic, mice_named=False)
stay_synthetic = {
    label: pd.concat([stay_rl.stay_by_run_length(rows.reset_index(drop=True)) for _, rows in table.groupby("mouse")], ignore_index=True)
    for label, table in long_inputs_synthetic.items()
}
long_recovery_synthetic = longk.recover_choice_kernel(
    synthetic.trials, alpha_ks=KERNEL_RECOVERY_RATES, beta_k=2.0, seed=STRATEGY_SEED,
    n_jobs=STRATEGY_JOBS, cache_dir=STRATEGY_CACHE_DIR,
)
persist_plots.plot_choice_kernel_figures(long_synthetic, stay_synthetic, kq_dir_synthetic, long_recovery_synthetic, mice_named=False)
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(strategy.kernel_parameters_table(comparison_synthetic["Q-Learner"]).round(3))
    display(kq_timescales_synthetic.round(3))
    display(strategy.ablation_summaries(comparison_synthetic).round(4))
    display(long_recovery_synthetic.round(3))
for name in [*kq_plots.FIGURE_NAMES, *persist_plots.FIGURE_NAMES]:
    show(kq_dir_synthetic / name)

**Reading the figures** (synthetic; the comparison and the kernel Q fits have 3 mice per source and show dots only; the
long-lag kernel and run-length curves have the 10 synthetic Q-learners and 10 agents per kind).

- *Kernel Q fits to the synthetic Q-learners.* The two plain learners (kappa 0) get alpha_k at its lower bound (0.001) with a
  kernel weight that has no effect (unidentified, flagged "alpha_k" at a bound); the sticky learner (kappa 1) is fitted as
  alpha_k 1.00 (a one-trial kernel), beta_k 0.99 and learning rates 0.31 and 0.30: the value-agent special case is recovered.
  The ablations cost nothing for these learners (all differences within 0.001 nats, except no kernel for the sticky one,
  +0.035). On the WSLS (Magnitude) source the kernel and forgetting help (+0.02 to 0.04).
- *Choice Kernel Long.* The simulated Q-learners have a sharp kernel (about 0.2 at lag 1, then negative values from
  value feedback), the WSLS agents 0.75 to 1.1 at lag 1 and about 0 after; WSLS (Habit) agents show a flat positive kernel
  (0.2 to 0.25 at lags 2 to 5). **The known-kernel recovery** (kernel-only agents, alpha_k 0.05, 0.1, 0.3, beta_k 2) is recovered
  at every lag within the se bands (alpha_k 0.3: 0.597 fitted against 0.600 true at lag 1).
- *Stay by run length.* The Q-learners rise with run length (0.58 to 0.91 after non-zero), the WSLS agents without a kernel
  stay flat (0.74 to 0.82), WSLS (Habit) rises like the Q-learners: the figure separates a slow kernel from its absence.


## 3. Real data

The drifting problems of `COHORTS`, pooled as in the other all-cohort analyses (problems are keyed
`(cohort, number within the drifting type)`). The model is fitted per mouse and validated by leaving
one problem out. A mouse with a single drifting problem or fewer than 300 trials cannot be validated;
it is skipped with a warning and listed.

In [ ]:
problems = pipe.load_drifting_problems(COHORTS, DATA_ROOT)
trials = drifting_trial_table(problems)
trials.groupby(["mouse", "problem"]).size().unstack(fill_value=0)

In [ ]:
real_selection = pipe.select_tail_decay(trials, TAIL_GRID, n_lags=N_LAGS, l2=L2)
real = real_selection.analysis
real_no_tail = pipe.analyse_trial_table(trials, n_lags=N_LAGS, tail_decay=None, l2=L2)
print("skipped:", real.skipped or "none")
print(f"chosen tail decay: {real_selection.decay}")
print(
    f"fraction of arm values exactly 0: {mvp.zero_fraction(real_no_tail.values):.3f} without the tail, "
    f"{mvp.zero_fraction(real.values):.3f} with it"
)
print(f"{len(real.designs)} analysed mice: {sorted(real.designs)}")
base = real_dir / "Choice Logit Weights"
plots.plot_kernels(real.weights, base)
show(base)

Weights are those of the full model fitted to all of a mouse's trials. Compare the reward kernel with
the Q-learner's from section 2: a positive, decaying kernel is what a learner that updates its values from
the rewards it receives looks like. The choice kernel shows perseveration (and, as explained above, part of
the correction for repeatedly chosen arms).

In [ ]:
base = real_dir / "Model Comparison"
plots.plot_model_comparison(real.comparison, base)
show(base)
real.comparison.pivot(index="mouse", columns="model", values="nats_per_choice")[list(cv.MODELS)].round(4)

### 3.1 Psychometrics: model value next to the task and believed values

Row 1 ranks the arms by the held-out model value (quantile bins per mouse); row 2 by the magnitudes
the ports offer; row 3 by the believed value (what an arm paid the last time the mouse chose it;
trials once all three arms of the problem have been chosen). All rows use the **same mice and
problems** as the model value, and each has the dashed baseline.

In [ ]:
real_bases = dict(
    model=(None, "Model Value", "Logit", N_BINS),
    task=(pipe.task_values(real), "Task Magnitude", "Pulses", None),
    believed=(pipe.believed_values(real), "Believed Value", "Pulses", None),
)
rows = []
for values, name, unit, n_bins in real_bases.values():
    curves, baselines = pipe.value_curves(real, n_bins, values, MIN_MICE)
    rows.append(
        plots.PsychometricRow(name, curves, plots.value_labels(name, unit, binned=n_bins is not None), baselines)
    )
base = real_dir / "Psychometric Curves"
plots.plot_psychometric_rows(
    rows,
    "Psychometric Curves: Model, Task and Believed Values",
    (
        "Rows rank the arms by the held-out model value (per-mouse quantile bins), by the magnitude "
        "each port offers, and by the last reward each arm paid (exact values).",
        "Row 3 counts trials once all three arms of the problem have been chosen.",
    ),
    base,
    MIN_MICE,
)
show(base)

### 3.2 Stay/switch curves: model value next to believed value

Rows by whether the previous choice was rewarded. First the model value, then the believed value, each
with its dashed baseline; each is followed by its switch-target figure (best alternative against the
gap to the other unchosen arm, switches only).

In [ ]:
for key, label, unit in (("model", "Model Value", "Logit"), ("believed", "Believed Value", "Pulses")):
    values, _, _, n_bins = real_bases[key]
    pairs = pipe.value_pairs(real, values)
    curves, baselines = mvp.stay_switch_curves_by_reward(
        pairs, pool_trials=POOL_TRIALS, n_bins=n_bins, min_mice=MIN_MICE
    )
    base = real_dir / f"Stay-Switch Curves ({label})"
    plots.plot_stay_switch_by_reward(
        curves, label, POOL_TRIALS, base,
        baselines_by_row=baselines, unit=unit, min_mice=MIN_MICE, binned=n_bins is not None,
    )
    show(base)

    target_curves, target_baselines = mvp.switch_target_curves_by_reward(pairs, n_bins=n_bins, min_mice=MIN_MICE)
    base = real_dir / f"Switch Target Curves ({label})"
    plots.plot_switch_targets_by_reward(
        target_curves, label, mvp.tied_switches_by_reward(pairs), base,
        baselines_by_row=target_baselines, unit=unit, min_mice=MIN_MICE, binned=n_bins is not None,
    )
    show(base)

### Fitted Q-learner: a positive control for 3.3 to 3.6

The win-stay/lose-shift agents are negative controls. The matching **positive control** is a Q-learner
(`value_agent`: learning rate, beta, stickiness, bias to the plus-shaped ports) **fitted to each mouse**
on all of its problems (several starts; the fits of the held-out model comparison are reused), then
played on that mouse's own problems under the cohort's task rules, exactly as the WSLS agents are
(`src/pipelines/fitted_q.py`). Its simulated mice carry the real mouse's name, so they take that
mouse's color. It shows what a value learner with *this mouse's* noise level (a low beta, strong
stickiness) looks like to every analysis below: the weak value weights of a mouse may be what such a
learner produces. It is not a null: the per-mouse calibration stays against the WSLS agents. It is
listed as **Q-Learner (Fitted)**; the synthetic **Q-Learner** of section 2 stays as it was.

**Schedule length.** The comparison columns of 3.3 to 3.6 (the three WSLS agents and the fitted
Q-learner) are simulated on schedules `CONTROL_LENGTH_FACTOR` times as long as the mouse's (each
problem's sessions repeated), so that a mouse with few trials does not leave its agents under the switch
regression's minimum of 100 switches, and the agents are less noisy: they show what each agent does in
principle. **The calibration bands, calibrated z and p (switch regression and second best) and the power
check use the mouse's own length** (factor 1), so that the null noise matches the mouse's.
Parameters at a bound of the fit are flagged.

**The second positive control** is the **Q-Learner (Kernel + Forgetting, Fitted)**: the same procedure with
the kernel Q-learner of 3.7 (asymmetric rates, forgetting, a multi-trial choice kernel), fitted to each mouse in
the held-out comparison and played on its own problems. It is the control for the question the first one
cannot answer: whether a *flat, slowly built* choice kernel plus forgetting reproduces the mice's weak value
weights. Wherever "Q-Learner (Fitted)" appears in 3.3 to 3.6 (switch regression, regression extensions, second
best, power check) this one appears next to it. **The fitted Q-learners' random streams differ from earlier runs**
(the stream index follows the number of null-agent kinds, which grew with WSLS (Habit)), so the earlier
simulated numbers of the first control change in the last digits.


In [ ]:
analyzed = trials[trials["mouse"].isin(sorted(real.designs))].reset_index(drop=True)
real_rules = nulls.task_rules_by_cohort(analyzed)  # the rules of the WSLS agents' simulations too
strategy_real = strategy.analyze_comparison_source(
    analyzed, real, n_starts=STRATEGY_STARTS, n_jobs=STRATEGY_JOBS, cache_dir=STRATEGY_CACHE_DIR
)
print("not compared (and so no fitted Q-learner):", strategy_real.skipped or "none")
q_fits = fitted_q.fits_of(strategy_real)
q_fitted_cohort = fitted_q.simulate_fitted_q_cohort(
    analyzed, q_fits, NULL_SEED + 10, real_rules, length_factor=CONTROL_LENGTH_FACTOR
)
display(fitted_q.parameters_table(q_fits).round(3))
# the second positive control: the Q-learner with kernel and forgetting, fitted to each mouse
kq_fits = fitted_q.fits_of(strategy_real, strategy_models.KERNEL_Q)
kq_fitted_cohort = fitted_q.simulate_fitted_q_cohort(
    analyzed, kq_fits, NULL_SEED + 10, real_rules, length_factor=CONTROL_LENGTH_FACTOR,
    model=strategy_models.KERNEL_Q,
)
display(fitted_q.parameters_table(kq_fits, strategy_models.KERNEL_Q).round(3))

### 3.3 Negative controls on the mice

The same four win-stay/lose-shift agents (section 2.7), fitted to each analysed mouse and played on
that mouse's own problems, then run through the logit: kernels, model comparison, psychometrics with
the baseline, and stay/switch and switch-target curves. Compare each row with the mice's figures above.

In [ ]:
null_real = run_null_controls(
    trials, real, "Mice", real_dir / "negative-controls",
    extra_cohorts={fitted_q.LABEL: q_fitted_cohort, fitted_q.CONTROLS[strategy_models.KERNEL_Q].label: kq_fitted_cohort}, rules=real_rules,
    length_factor=CONTROL_LENGTH_FACTOR,
)

**Reading the figures** (cohorts 1 and 3, 14 mice; agents on schedules `CONTROL_LENGTH_FACTOR` times
the mouse's). Every likelihood is leave-one-problem-out on the same folds. The choice logit has the lowest
negative log likelihood per trial for all 14 mice (0.42 to 0.80). Of the agents, WSLS (Magnitude) is best for 9 mice,
WSLS (Recency) for 3 (MY_175_N, MY_66_LR, MY_66_N, each within 0.005 of WSLS (Magnitude) except MY_66_N) and
WSLS (Graded) for 1 (MY_66_LL, 0.880 against 0.905). The rise of P(best) beyond the baseline is 0.094 for the
mice against 0.047, 0.037, 0.095 and 0.122 for WSLS, WSLS (Recency), WSLS (Magnitude) and WSLS (Graded) and 0.062 for
the fitted Q-learner. The switch-target excess is 0.035 (rewarded) and 0.020 (unrewarded) for the mice, between
-0.011 and 0.030 for the four agents and 0.079 and 0.056 for the fitted Q-learner: the mice sit between the
agents and the positive control, and the five-lag blind spot of the baseline (see 2.7) applies to all of them.
WSLS (Graded) per mouse: `w0` from -0.49 to 0.62, `w_m` from 0.28 to 0.73 (mean 0.47), `p_recent` from 0.55 to
0.73 (mean 0.64).

### 3.4 Regression Extensions

The four analyses of 2.8 on the mice, next to the controls: the **Q-learners** of section 2 (the
analysis of 2.8, reused) and the **win-stay/lose-shift agents** fitted to each mouse of section 3.3
(the same simulated agents, reused). Read each mouse's gains, weights and calibration against the
controls; the mice are as few as the staged cohorts hold.

In [ ]:
ext_real = ext.analyse_sources(ext.comparison_inputs("Mice", trials, null_real), EXT_JOBS, EXT_CACHE_DIR)
real_sources = {
    "Mice": ext_real["Mice"],
    "Q-Learner": ext_synthetic["Q-Learner"],
    **{label: source for label, source in ext_real.items() if label != "Mice"},
}
tables_real, summaries_real = report_regression_extensions(real_sources, real_dir / "regression-extensions")

**Reading the figures** (one mouse, MY_175_L, so no se bars and no test; sources with fewer than 6 mice show the mouse only, the Q-learner column is the 10 simulated mice of 2.8). Believed value adds nothing to the full model (-0.0005 nats per choice; weight 0.011 logit per pulse), and replacing the reward lags with it costs 0.0126, more than for the Q-learner (0.0073 cost) and about like WSLS (Magnitude) (0.0088 for the fitted agent). Its never-visited weight is -0.78 logit: an arm not yet visited in a problem is chosen less than an arm that last paid 0. Pulse is slightly preferred to binary (-0.0021, binary tail decay 0.95 chosen by the binary model's own likelihood) and adding binary lags to the pulse lags does not help (-0.0024), as for the Q-learner and WSLS (Magnitude) rather than the two magnitude-blind agents (+0.0072, +0.0050 with binary preferred). Dropping the tower bias *improves* held-out fit (unique tower -0.020): with few repeated towers across a mouse's problems the fold's tower weights do not transfer. Unique reward is -0.0004, unique choice 0.039 and shared 0.323; the agents fitted to this mouse have unique reward 0.010 to 0.014 and shared 0.18 to 0.20. Calibration is not predictive quality: the full model's mean |observed - predicted| is 0.026 against a noise floor of 0.010 and the choice-history-only model's 0.046 against 0.011, so both are miscalibrated beyond sampling noise, the choice-only model more; the mouse's held-out nats per choice are 0.771 (full) and 0.765 (choice only), i.e. here the choice-only model predicts about as well. One mouse licenses no group statement; these are the numbers the cohort run will fill in.

### 3.5 Switch Regression

The switch regression of 2.9 on the mice, next to the Q-learners of section 2 (reused) and the
four win-stay/lose-shift agents fitted to each mouse in 3.3 (reused); the stress agents are in
2.9. Each mouse is read against its own null (gray band; calibrated z and p in the tables and the
caption), against its fitted WSLS (Magnitude), WSLS (Recency) and WSLS (Graded) agents with
`SWITCH_NULL_REPLICATES` replicates each. Readouts: the believed value and the true value after
non-zero; the true value after zero cannot separate a value learner from the null agents. The real
data are the cohorts of `COHORTS`; Meg will run cohorts 1, 3 and 4 (cohort 4 is
Ruby's mice). A mouse with fewer than 100 switches in either condition is skipped with a warning.

In [ ]:
switch_real = switch.analyze_switch_sources(
    ext.comparison_inputs("Mice", trials, null_real),
    trace_grid=SWITCH_GRID, firth=SWITCH_FIRTH, n_jobs=EXT_JOBS, cache_dir=SWITCH_CACHE_DIR,
)
switch_real_sources = {
    "Mice": switch_real["Mice"],
    "Q-Learner": switch_synthetic["Q-Learner"],
    **{label: source for label, source in switch_real.items() if label != "Mice"},
}
print("skipped:", {k: s.skipped for k, s in switch_real.items() if s.skipped} or "none")
# each mouse against its own fitted WSLS (Magnitude) and WSLS (Recency), played on its own task
switch_real_calibration = switch.calibrate_source(
    trials, switch_real["Mice"], null_real.controls.fits, replicates=SWITCH_NULL_REPLICATES,
    seed=SWITCH_SEED, n_jobs=EXT_JOBS, cache_dir=SWITCH_CACHE_DIR,
)
switch_summary_real, switch_mice_real = report_switch_regression(
    switch_real_sources, real_dir / "switch-regression", {"Mice": switch_real_calibration}
)

**Reading the figures** (one mouse, MY_175_L, so there is no mean, se or test across mice; the
Q-learner column is the 10 simulated mice of 2.9). 340 switches after zero and 547 after non-zero, in
6 problems; trace decay 0.7. Calibrated against its fitted WSLS (Magnitude) and WSLS (Recency), 100
replicates each.

- *No value-guided switching.* The value weights lie inside the mouse's own null (gray bands):
  believed value +0.079 after zero (calibrated z 1.02 and 1.26 against the two agents, p 0.30 and 0.22)
  and +0.046 after non-zero (z 1.02 and 1.25, p 0.30 and 0.23); true value after non-zero +0.023
  (z 0.51 and 0.63, p 0.68 and 0.55). True value after zero (-0.010, p 0.80 and 0.90) is not a
  readout (see 2.9). The model value (+0.09 and +0.15 per logit) is not calibrated.
- *The held-out gains are below the null.* Adding any value makes this mouse's held-out switch targets
  worse to predict (-0.0045 to -0.0166 nats per switch), and more so than for its value-blind agents:
  calibrated z -2.0 to -6.8 for the believed value and the true value after non-zero (p 0.01 to 0.09).
  That is not evidence for value; it suggests the value weight fitted on the other problems does not
  carry over to the held-out one for this mouse (problems differ), and is worth checking per problem
  once there are more mice.
- *What carries the switches.* The trace weight is 0.65 after zero and 0.20 after non-zero, recency
  about 0 (history figure).
- With one mouse this is a description, not a group result. Meg will run cohorts 1, 3 and 4; from 6
  mice the captions also give the Wilcoxon test of the calibrated z across mice.

### 3.6 Strategy Tests

The two strategy tests of 2.10 on the mice. Second best: the mice, the synthetic Q-learners of section 2,
the four WSLS agents and the fitted Q-learner (3.3, reused) as sources; each mouse is calibrated against
its own fitted WSLS (Magnitude), WSLS (Recency) and WSLS (Graded) agents with `SECOND_BEST_REPLICATES` replicates
each (the fitted Q-learner is a positive control, not a null). Model comparison: the mice, next to the
synthetic comparison of 2.10. **Model recovery is run on the mice's own schedules and fitted parameters
first**; read its confusion matrix before the comparison. Meg will run cohorts 1, 3 and 4 (cohort 4 is
Ruby's mice); a source with fewer than 6 mice shows its mice only. The power check below the figures
asks of each mouse whether a Q-learner with that mouse's fitted parameters would have shown a value
weight above the mouse's own (switch regression of 3.5).

Cost, measured on cohorts 1 and 3 (14 mice, 4 workers, `STRATEGY_STARTS` Q starts, cold caches): the cell
below (second best with its calibration, the model comparison and the recovery with one replicate per model)
took 47 minutes, most of it the recovery, which refits four models on every simulated mouse; the power check took
5 minutes. With warm caches the cell takes seconds. The whole notebook took 107 minutes. **With the kernel Q-learner, its four ablations, WSLS (Habit) and their recovery, the cell below and the cells around it take much longer** (measured on 14 mice, 2 workers, cold caches): the comparison with the fitted controls 50 minutes, the recovery of six models on all 14 mice 2 hours 10 minutes, the extension and switch cells 23 and 22 minutes, the whole notebook about 4.6 hours; `RECOVERY_REAL_MICE` limits the recovery to the first mice.

In [ ]:
strategy_dir_real = real_dir / "strategy-tests"
second_inputs_real = {label: table for label, (table, _) in ext.comparison_inputs("Mice", analyzed, null_real).items()}
second_real = second.analyze_second_best_sources(
    {"Mice": second_inputs_real.pop("Mice"), "Q-Learner": synthetic.trials, **second_inputs_real}
)
print("skipped:", {k: s.skipped for k, s in second_real.items() if s.skipped} or "none")
second_calibrations_real = {
    "Mice": second.calibrate_second_best(
        analyzed, second_real["Mice"], null_real.controls.fits, replicates=SECOND_BEST_REPLICATES,
        rules=real_rules, seed=STRATEGY_SEED, n_jobs=STRATEGY_JOBS, cache_dir=STRATEGY_CACHE_DIR,
    )
}
recovery_mice_real = sorted(strategy_real.fits)[:RECOVERY_REAL_MICE]
recovery_real = strategy.recover_models(
    first_mice(analyzed, RECOVERY_REAL_MICE),
    strategy_real._replace(fits={m: strategy_real.fits[m] for m in recovery_mice_real}), rules=real_rules, replicates=RECOVERY_REPLICATES, seed=STRATEGY_SEED,
    n_jobs=STRATEGY_JOBS, cache_dir=STRATEGY_CACHE_DIR,
)
print(*strategy_plots.recovery_statement(recovery_real), sep="\n")
comparison_real = {"Mice": strategy_real, **comparison_synthetic}
report_strategy_tests(
    second_real, second_calibrations_real, comparison_real, recovery_real, strategy_dir_real
)

In [ ]:
power_real = fitted_q.power_check(
    analyzed, switch_real["Mice"], q_fits, replicates=POWER_REPLICATES, rules=real_rules,
    seed=SWITCH_SEED, n_jobs=STRATEGY_JOBS, cache_dir=STRATEGY_CACHE_DIR,
)
print(f"power check: share of {POWER_REPLICATES} fitted-Q replicates per mouse whose value weight exceeds the mouse's own")
display(power_real.round(3))
power_kernel_real = fitted_q.power_check(
    analyzed, switch_real["Mice"], kq_fits, replicates=POWER_REPLICATES, rules=real_rules,
    seed=SWITCH_SEED, n_jobs=STRATEGY_JOBS, cache_dir=STRATEGY_CACHE_DIR, model=strategy_models.KERNEL_Q,
)
print(f"power check of the kernel + forgetting Q-learner ({POWER_REPLICATES} replicates per mouse)")
display(power_kernel_real.round(3))

**Reading the figures** (cohorts 1 and 3, 14 mice; mean ± se across mice; Wilcoxon p across mice; the
simulated comparison sources have 3 mice and show dots only).

- *Recovery first.* Model recovery on the mice's own schedules (14 mice, six models, one replicate) is **poor for
  WSLS (Magnitude) (64% won by the model that generated them), Heuristic+ (57%) and the Q-learner (71%)**, and good for
  WSLS (Graded) (93%), the Q-learner (Kernel + Forgetting) (93%) and WSLS (Habit) (100%). The confusions are nesting:
  Heuristic+ mice are won by WSLS (Magnitude) in 29%; WSLS (Magnitude) mice are won by Heuristic+ (14%) or WSLS (Habit)
  (14%); a Q-learner mouse is won by the kernel Q-learner in 29% (it contains the Q-learner up to the kernel). Parameters
  recovered: `v` (correlation 0.94), the Q learning rate 0.98, `p_recent` 0.95, and for the kernel Q-learner alpha_pos
  0.93, zeta 0.96, alpha_k 0.96, beta 0.92, beta_k 0.97 and WSLS (Habit) w_h 0.98; **poorly** `u` (0.67) and the kernel
  Q-learner's alpha_neg (0.61; most mice sit at its bound of 1, see 3.7).
- *Second best.* After a non-zero payout the mice choose the second best in 0.459 ± 0.018 of the leave-best
  switches, the agents in 0.501 (WSLS), 0.487 (Recency), 0.480 (Magnitude) and 0.488 (Graded), the fitted Q-learner
  in 0.540 ± 0.017 and the synthetic Q-learners in 0.582. The mice are below 0.5 and not above their own nulls:
  calibrated z against WSLS (Magnitude) is -0.89 ± 0.45 (second best, p 0.12), -0.42 ± 0.78 (believed-better after
  non-zero, p 0.63) and -1.13 ± 0.59 (truly better after non-zero, p 0.10); against WSLS (Graded) the largest is
  believed-better after zero, 1.21 ± 0.51 (p 0.058, one of 24 uncorrected tests). There is no sign of a switch
  towards the better alternative beyond what recency and graded staying produce. The fitted Q-learner, which does
  switch on value, is above the mice on every measure after non-zero, so the test would have seen it. Leave-best after
  zero is empty for every source.
- *Model comparison.* Mean paired difference in held-out nats per choice (negative: first better): Heuristic+ minus
  Q-learner -0.0092 ± 0.0040 (p 0.09), WSLS (Magnitude) minus Q-learner -0.0080 ± 0.0047 (p 0.24), Heuristic+ minus
  WSLS (Magnitude) -0.0012 ± 0.0015 (p 0.90), WSLS (Graded) minus Q-learner +0.0021 ± 0.0042 (p 0.67), WSLS (Graded)
  minus WSLS (Magnitude) +0.0101 ± 0.0033 (p 0.013; graded fits worse in 13 of 14 mice). **The two new models are far
  ahead of those:** the Q-learner (Kernel + Forgetting) minus WSLS (Magnitude) is -0.0863 ± 0.0121 (p 0.0001, better in
  all 14 mice), minus the Q-learner -0.0943 ± 0.0106 (p 0.0001, all 14), and WSLS (Habit) minus WSLS (Magnitude) is
  -0.0806 ± 0.0076 (p 0.0001, all 14). WSLS (Habit) minus the kernel Q-learner is +0.0057 ± 0.0064 (p 0.17: the habit
  agent fits worse in 9 of 14, not distinguishable). Wins: the kernel Q-learner 9, WSLS (Habit) 5, no other model any.
  Given the recovery (both new models 93% to 100%) a win by either is evidence for it against the older four.
- *Power check.* For 11 of the 13 evaluable mice at least 86% of the fitted Q-learner's replicates show a larger
  believed-value weight after non-zero than the mouse (the mouse's is below what its own fitted value learner
  would produce). The exceptions are MY_175_L (12%) and MY_66_LR (8%), whose fitted learning rates are about 0.005:
  a learner that does not learn has no power here. MY_66_LL (741 trials) had no replicate with enough switches at
  its own length and MY_175_N 17 of 50.
- *Skipped.* With `CONTROL_LENGTH_FACTOR` 2 no mouse is skipped anywhere in 3.3 to 3.6. The calibration nulls
  (mouse's own length) lose replicates for MY_175_N (86, 80 and 90 of 100 for WSLS (Magnitude), (Recency) and
  (Graded)), MY_66_LL (52, 57 and 39) and MY_66_LR (5, 2 and 2).
- *Fitted Q-learners.* No parameter of the first control is at a bound; learning rates are 0.004 to 0.011 for
  MY_175_L, MY_174_LR and MY_66_LR, 0.18 to 0.73 for the others. The kernel + forgetting control is described in 3.7.
  Its second-best and switch-target numbers are the new rows of the tables above: after a non-zero payout it chooses the
  second best in 0.497 ± 0.010 of leave-best switches (mice 0.459 ± 0.018, the first fitted Q-learner 0.537 ± 0.012,
  WSLS (Habit) 0.469 ± 0.008), and believed-better in 0.507 ± 0.012 (mice 0.493, first control 0.595). The fitted Q-learners'
  random streams differ from earlier runs (see the fitted-Q section), so the first control's numbers moved in the last digit.
- *Power check of the kernel + forgetting control.* The share of its replicates whose believed-value weight after non-zero
  exceeds the mouse's is at most 0.80 for every mouse (0.76, 0.02, 0.00, 0.80, 0.10, 0.62, 0.90, 0.88, 0.28, 0.02, 0.52, 0.06,
  0.00, 0.00 in the order of the mice; 0.90 and 0.88 for MY_66_N and MY_66_R, whose mice have the smallest weights), against
  at least 0.86 for 11 of 13 mice for the first control. The mice's value-guided switching is of the size that a Q-learner with a
  slow kernel and forgetting produces, which the first control could not show.


### 3.7 Kernel Q-Learner and the Flat Choice Kernel

The **Q-Learner (Kernel + Forgetting)** and **WSLS (Habit)** are in the comparison and the recovery of 3.6
(the same folds, the same runs). Here: what the four ablations cost, the fitted parameters per mouse, the
held-out trace-decay profile of WSLS (Magnitude) (`Choice Kernel Decay`), the choice kernel to lag 15 with and
without a choice tail (`Choice Kernel Long`, with its known-kernel recovery) and P(stay) by run length
(`Stay by Run Length`), each for the mice and the agents played on the mice's own schedules.
Figures go to `kernel-q/`. See `docs/issues/feat-kernel-q.md` for the model, the equation mapping and the
two ways it differs from the paper (Q carries across sessions; plus-port bias instead of left/right).

In [ ]:
kq_dir_real = real_dir / "kernel-q"
kq_profile_real = strategy.decay_profile_table(
    analyzed, sorted(strategy_real.fits), n_jobs=STRATEGY_JOBS, cache_dir=STRATEGY_CACHE_DIR
)
long_inputs_real = {"Mice": analyzed, **second_inputs_real}  # agents and fitted Q-learners, as 3.6
long_real = {
    label: longk.analyze_long_kernel(table, n_lags=KERNEL_LONG_LAGS, n_jobs=STRATEGY_JOBS, cache_dir=STRATEGY_CACHE_DIR)
    for label, table in long_inputs_real.items()
}
kq_timescales_real = strategy.timescale_table(strategy_real, long_real["Mice"].weights)
kq_plots.plot_kernel_q_figures(comparison_real, kq_profile_real, kq_timescales_real, kq_dir_real)
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    print("fitted Q-Learner (Kernel + Forgetting) per mouse")
    display(strategy.kernel_parameters_table(strategy_real).round(3))
    print("kernel timescale per mouse: fitted alpha_k and decay 1 - alpha_k of the kernel Q-learner and WSLS (Habit), and the long-lag fit")
    display(kq_timescales_real.round(3))
    print("ablations: held-out nats per choice of each ablation minus the full model (positive: ablation fits worse)")
    display(strategy.ablation_summaries(comparison_real).round(4))
    print("WSLS (Magnitude) held-out nats per choice by trace decay (0: no trace)")
    display(kq_profile_real.pivot(index="mouse", columns="decay", values="nats_per_choice").round(4))
for name in kq_plots.FIGURE_NAMES:
    show(kq_dir_real / name)

In [ ]:
# Choice kernel to lag 15: tail or no tail, and the agents refitted with the same design.
persist_dir_real = real_dir / "choice-kernel"
beta_k_real = {m: f[strategy_models.KERNEL_Q][kq_agent.INDEX["beta_k"]] for m, f in strategy_real.fits.items()}
long_recovery_real = longk.recover_choice_kernel(
    analyzed, alpha_ks=KERNEL_RECOVERY_RATES, beta_k=beta_k_real, rules=real_rules, n_lags=KERNEL_LONG_LAGS,
    seed=STRATEGY_SEED, n_jobs=STRATEGY_JOBS, cache_dir=STRATEGY_CACHE_DIR,
)
stay_real = {
    label: pd.concat([stay_rl.stay_by_run_length(rows.reset_index(drop=True)) for _, rows in table.groupby("mouse")], ignore_index=True)
    for label, table in {"Mice": analyzed, **second_inputs_real}.items()
}
persist_plots.plot_choice_kernel_figures(long_real, stay_real, persist_dir_real, long_recovery_real)
print("held-out nats per choice of the long-lag logit, without and with the choice tail")
display(long_real["Mice"].comparison.pivot(index="mouse", columns="model", values="nats_per_choice").round(4))
print("known-kernel recovery (fitted mean against the generating weight)")
display(long_recovery_real.round(3))
print("P(stay) slope per run-length bin, by source and payout (mean across the source's mice)")
slopes_real = pd.concat({k: stay_rl.run_length_slopes(t) for k, t in stay_real.items()}, names=["source"]).reset_index(level=0)
display(slopes_real.groupby(["source", "payout"])["slope"].agg(["mean", "std", "count"]).round(4))
for name in persist_plots.FIGURE_NAMES:
    show(persist_dir_real / name)

**Reading the figures** (cohorts 1 and 3, 14 mice; mean ± se across mice; Wilcoxon p; simulated sources have
fewer than 6 mice and show dots only; figures in `real/kernel-q/` and `real/choice-kernel/`).

- *Fitted parameters* (Q-learner with kernel + forgetting; alpha_pos, alpha_neg, zeta, alpha_k, beta, beta_k, mean ± se):
  0.31 ± 0.04, 0.94 ± 0.03, 0.41 ± 0.09, 0.26 ± 0.04, 0.71 ± 0.05, 1.59 ± 0.06. **Nine mice have alpha_neg at its bound of 1**
  (and MY_175_L has zeta at 0): losses are learned in one trial and unchosen values are largely forgotten, a regime in which
  the asymmetry is poorly identified (recovery of alpha_neg 0.61). The kernel rate alpha_k is 0.12 to 0.63 (decay 0.37 to 0.88),
  and beta_k (1.2 to 1.8) is larger than beta (0.4 to 1.2): the kernel weighs more than the value in the choice.
  Per-mouse values are in the printed table and in the caption of Kernel Q Parameters.
- *Ablations* (ablation minus full, held-out nats per choice, positive: the removed piece helps): no kernel +0.0296 ± 0.0029
  (all 14, p 0.0001), kernel only +0.0145 ± 0.0024 (all 14), no forgetting +0.0138 ± 0.0029 (13 of 14, p 0.0002), no
  asymmetry +0.0036 ± 0.0010 (12 of 14, p 0.004). Each piece helps; the kernel the most, asymmetry the least. The one-trial
  Q-learner of 3.6 is 0.094 worse than the full model, so most of that gap is the multi-trial kernel, not the other pieces.
- *Trace decay of WSLS (Magnitude).* Held-out nats are lowest at a decay of 0.9 for six mice, 0.8 for four, 0.7
  for three and 0.95 for one of the 14 (profile table above); no trace is worse than 0.9 by 0.005 to 0.031 for every mouse. The fitted 1 - alpha_k of the
  kernel Q-learner is 0.74 ± 0.04, that of WSLS (Habit) (rate from a grid) 0.67 ± 0.02, and the geometric decay of the choice
  logit's lag weights 0.76 ± 0.05 (timescale 0.5 to 15 trials, median 4.7). The three estimates agree on a decay of about 0.7 to 0.8
  per trial, a timescale of 3 to 5 trials (half-life 2 to 3.5): slower than a one-trial stickiness, faster than the WSLS trace of 0.9.
- *Choice Kernel Long.* The mice's choice weights are flat: about 0.2 to 0.25 for lags 1 to 5 and falling to 0 by lag 8 to 10.
  The agents' sharp kernels (WSLS and WSLS (Recency) 0.7 to 1.1 at lag 1 and about 0 after; the fitted Q-learner 1.1 at lag 1, then slightly
  negative) do not look like that, while the fitted kernel + forgetting Q-learner (0.35 at lag 1, decaying to 0 by lag 10) and
  WSLS (Habit) (0.3 at lag 1, rising to 0.5 at lag 2, decaying) do. The choice tail (decay 0.9, beyond lag 15) does not help held out:
  nats with the tail minus without are +0.0001 to +0.0086 (worse or equal in 13 of 14 mice, better only for MY_71_R by 0.0008).
  The known-kernel recovery (kernel-only agents on the mice's schedules, alpha_k 0.05, 0.1, 0.3) is recovered to within the
  se bands at every lag (e.g. alpha_k 0.3: fitted 0.475 against 0.478 at lag 1, 0.312 against 0.335 at lag 2), so a slow kernel
  would be seen as one.
- *Stay by run length.* P(stay) rises with run length in the mice after both payouts: slope per bin step 0.083 ± 0.020 (std across
  mice, non-zero) and 0.055 ± 0.018 (zero) [std, not se]. Plain WSLS and WSLS (Recency) are flat (0.001), WSLS (Magnitude) 0.006,
  WSLS (Graded) 0.011 and the first fitted Q-learner 0.023 (non-zero), so the rise needs a slowly built kernel. The kernel +
  forgetting Q-learner (0.078, 0.064) and WSLS (Habit) (0.068, 0.095) rise as the mice do. After zero payout WSLS (Habit) rises too
  fast (0.095 against the mice's 0.055).
- *Models.* WSLS (Habit) and the kernel Q-learner both beat WSLS (Magnitude) by about 0.08 nats (3.6) and cannot be told apart
  from each other on held-out fit (+0.0057 ± 0.0064). The mice's fitted habit rate alpha_k is 0.2 to 0.5 (0.3 for 10 mice, 0.5 for 3, 0.2 for 1).
- *Does habit alone explain the value weight?* No. The fitted WSLS (Habit) is a fourth null of the switch regression and
  the second-best test (same 100 replicates per mouse on its own task, same analysis; the drawn band stays WSLS
  (Magnitude), whose null means are within 0.015 of the habit agent's). The habit source's raw value weights are 0:
  believed 0.021 ± 0.013 after zero and 0.003 ± 0.010 after non-zero, true 0.008 ± 0.014 and -0.005 ± 0.010 (no Wilcoxon
  p below 0.17), against the mice's 0.088, 0.087, 0.016 and 0.048. Calibrated z of the mice against the habit null (mean ± se,
  14 mice, Wilcoxon p): believed after non-zero **2.30 ± 0.52 (p 0.0006)**, believed after zero 1.21 ± 0.36 (p 0.013), true
  after non-zero 1.37 ± 0.46 (p 0.020), true after zero 0.24 ± 0.37 (p 0.58); against WSLS (Magnitude) the same readouts are
  2.20 ± 0.50, 1.08 ± 0.35, 1.47 ± 0.50 and 0.27 ± 0.34. Second best against the habit null: second best after non-zero
  -0.005 ± 0.454 (p 0.90), believed-better 1.12 ± 0.54 after zero and 0.84 ± 0.80 after non-zero, truly better 0.45 ± 0.42 and
  0.11 ± 0.58 (p 0.54 and 1.00): none different from 0. Mice that beat the habit null on believed value after non-zero (empirical
  p < 0.05): MY_66_LL (0.012), MY_71_N (0.030), MY_71_R, MY_72_L, MY_72_N and MY_72_R (0.0099, the floor of 100 replicates);
  on true value after non-zero: MY_66_LL (0.012), MY_71_R, MY_72_N and MY_72_R. MY_174_R (0.069) is borderline. The
  kernel + forgetting Q-learner source (a habit-plus-value learner, a comparison and not a null) has raw weights between the two:
  believed 0.068 ± 0.020 after zero and 0.056 ± 0.017 after non-zero, true 0.038 ± 0.023 and 0.038 ± 0.012, against the mice's 0.087
  and the first fitted Q-learner's 0.218 ± 0.033 (believed, after non-zero).
- *Limits.* The tail of the choice kernel, the pooling of non-zero payouts in the run-length curves (the magnitude mix differs
  between short and long runs, the agents carry the same mix) and the grid of habit rates are decisions, documented in
  `docs/issues/feat-kernel-q.md`.


### Notes and limits

- $\hat V$ is a *linear, finite-memory* summary of a mouse's reward history ($K$ = `N_LAGS` trials plus
  the decayed tail), chosen because it can be read straight from choices. It is not a Q-learner's value
  (flatter kernel; part of the Q-learner's forgetting sits in the choice weights); a real mouse need not
  be a Q-learner.
- The model value is in logits. Quantile bins place each bin at its mean value, so the x axis has the
  value's units, but the scale differs between mice (each mouse has its own weights); the x of a
  curve is the mean over mice of the bin means.
- Tower biases of towers that a fold's training problems never offered stay at 0 (the penalty), so
  the model comparison's tower step only helps where towers repeat across a mouse's problems.
- Real-data figures are as good as the number of mice. The staged cohort snapshots used here hold few
  drifting mice, and a mean from fewer than `MIN_MICE` mice is only drawn when the data have fewer
  mice than that in total; with one mouse the curves have no standard error.